# Google Drive Data Streaming

This notebook provides functions to stream data from Google Drive.

In [94]:
# Install required packages (uncomment if needed)
# !pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

In [95]:
import io
import os
from typing import Generator, Optional, Union

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

In [96]:
oauth_credentials_json = 'client_secret_1040199529314-06cjn46g0c3jg509kk9lrgl9raaskb7m.apps.googleusercontent.com.json'

In [97]:
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']


def get_google_drive_service(credentials_path: str = 'credentials.json',
                              token_path: str = 'token.json'):
    """
    Authenticate and return a Google Drive service object.
    
    Args:
        credentials_path: Path to the OAuth 2.0 credentials JSON file
                          (download from Google Cloud Console)
        token_path: Path to store/retrieve the user's access token
    
    Returns:
        Google Drive API service object
    """
    creds = None
    
    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(credentials_path, SCOPES)
            creds = flow.run_local_server(port=0)
        
        with open(token_path, 'w') as token:
            token.write(creds.to_json())
    
    return build('drive', 'v3', credentials=creds)

In [98]:
# =============================================================================
# DEMO: Set your Google Drive URL here
# =============================================================================

google_drive_url = 'https://drive.google.com/drive/u/1/folders/1atBYNLP5UxUx5keXja5wcUbduYPBD8sn'

# Maximum data to stream (set to None for unlimited)
max_gb = 1  # Stream up to 1 GB

# Step 1: Authenticate with Google Drive
CREDENTIALS_FILE = oauth_credentials_json
TOKEN_FILE = 'gdrive_token.json'  # Using different name to avoid directory conflict

service = get_google_drive_service(credentials_path=CREDENTIALS_FILE, token_path=TOKEN_FILE)
print("Authenticated successfully!")

Authenticated successfully!


In [99]:
GOOGLE_DOCS_EXPORT_MIMETYPES = {
    'application/vnd.google-apps.document': ('application/pdf', '.pdf'),
    'application/vnd.google-apps.spreadsheet': ('text/csv', '.csv'),
    'application/vnd.google-apps.presentation': ('application/pdf', '.pdf'),
    'application/vnd.google-apps.drawing': ('image/png', '.png'),
}

def stream_file_from_drive(service, file_id: str, 
                           mime_type: str = None,
                           chunk_size: int = 1024 * 1024) -> Generator[bytes, None, None]:
    """
    Stream a file from Google Drive in chunks.
    
    Args:
        service: Google Drive API service object
        file_id: The ID of the file to download (from the Google Drive URL)
        mime_type: The mimeType of the file (needed for Google Docs export)
        chunk_size: Size of each chunk in bytes (default: 1MB)
    
    Yields:
        Chunks of file data as bytes
    """
    # Check if this is a Google Docs file that needs export
    if mime_type in GOOGLE_DOCS_EXPORT_MIMETYPES:
        export_mime, _ = GOOGLE_DOCS_EXPORT_MIMETYPES[mime_type]
        request = service.files().export_media(fileId=file_id, mimeType=export_mime)
    else:
        request = service.files().get_media(fileId=file_id)
    
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request, chunksize=chunk_size)
    
    done = False
    while not done:
        status, done = downloader.next_chunk()
        chunk_data = buffer.getvalue()
        buffer.seek(0)
        buffer.truncate(0)
        if chunk_data:
            yield chunk_data
        if status:
            print(f"Download progress: {int(status.progress() * 100)}%")

In [100]:
def stream_text_lines_from_drive(service, file_id: str,
                                  encoding: str = 'utf-8',
                                  chunk_size: int = 1024 * 1024) -> Generator[str, None, None]:
    """
    Stream a text file from Google Drive line by line.
    
    Args:
        service: Google Drive API service object
        file_id: The ID of the file to download
        encoding: Text encoding (default: utf-8)
        chunk_size: Size of each download chunk in bytes
    
    Yields:
        Individual lines from the text file
    """
    buffer = ''
    
    for chunk in stream_file_from_drive(service, file_id, chunk_size):
        buffer += chunk.decode(encoding)
        lines = buffer.split('\n')
        buffer = lines.pop()  # Keep incomplete line in buffer
        
        for line in lines:
            yield line
    
    # Yield any remaining content
    if buffer:
        yield buffer

In [101]:
def list_files_in_folder(service, folder_id: Optional[str] = None,
                          page_size: int = 100,
                          recursive: bool = False) -> Generator[dict, None, None]:
    """
    Stream file metadata from a Google Drive folder.
    
    Args:
        service: Google Drive API service object
        folder_id: The ID of the folder (None for root)
        page_size: Number of files per API request
        recursive: If True, also list files in nested subfolders
    
    Yields:
        File metadata dictionaries with 'id', 'name', 'mimeType', 'path'
    """
    def _list_folder(fid, path=""):
        query = f"'{fid}' in parents" if fid else None
        page_token = None
        
        while True:
            results = service.files().list(
                q=query,
                pageSize=page_size,
                pageToken=page_token,
                fields="nextPageToken, files(id, name, mimeType, size)"
            ).execute()
            
            for file in results.get('files', []):
                file['path'] = f"{path}/{file['name']}" if path else file['name']
                
                if file.get('mimeType') == 'application/vnd.google-apps.folder':
                    if recursive:
                        # Recurse into subfolder
                        yield from _list_folder(file['id'], file['path'])
                else:
                    yield file
            
            page_token = results.get('nextPageToken')
            if not page_token:
                break
    
    yield from _list_folder(folder_id)

In [102]:
def extract_file_id(drive_url: str) -> str:
    """
    Extract the file ID from a Google Drive URL.
    
    Args:
        drive_url: A Google Drive sharing URL
    
    Returns:
        The file ID string
    """
    import re
    
    patterns = [
        r'/file/d/([a-zA-Z0-9_-]+)',  # /file/d/ID format
        r'id=([a-zA-Z0-9_-]+)',        # ?id=ID format
        r'/folders/([a-zA-Z0-9_-]+)',  # /folders/ID format
    ]
    
    for pattern in patterns:
        match = re.search(pattern, drive_url)
        if match:
            return match.group(1)
    
    # Assume the input is already a file ID
    return drive_url

## Folder Streaming Functions

In [103]:
def stream_folder(service, folder_id: str,
                  max_bytes: Optional[int] = None,
                  file_types: Optional[list[str]] = None,
                  recursive: bool = True,
                  chunk_size: int = 1024 * 1024) -> Generator[tuple[dict, bytes], None, None]:
    """
    Stream all files from a Google Drive folder with optional size limit.
    
    Args:
        service: Google Drive API service object
        folder_id: The ID of the folder to stream from
        max_bytes: Maximum total bytes to stream (e.g., 5 * 1024**3 for 5GB).
                   None for unlimited.
        file_types: List of file extensions to include (e.g., ['.csv', '.txt']).
                    None for all files.
        recursive: If True, include files in nested subfolders (default: True)
        chunk_size: Size of each download chunk in bytes
    
    Yields:
        Tuples of (file_metadata, chunk_bytes) for each chunk of each file
    """
    total_bytes = 0
    
    for file_info in list_files_in_folder(service, folder_id, recursive=recursive):
        # Skip folders
        if file_info.get('mimeType') == 'application/vnd.google-apps.folder':
            continue
        
        # Filter by file type if specified
        if file_types:
            file_name = file_info.get('name', '')
            if not any(file_name.lower().endswith(ext.lower()) for ext in file_types):
                continue
        
        print(f"Streaming: {file_info.get('path', file_info['name'])}")
        
        try:
            for chunk in stream_file_from_drive(service, file_info['id'], 
                                                 mime_type=file_info.get('mimeType'),
                                                 chunk_size=chunk_size):
                # Check if we've hit the byte limit
                if max_bytes and total_bytes + len(chunk) > max_bytes:
                    remaining = max_bytes - total_bytes
                    if remaining > 0:
                        yield file_info, chunk[:remaining]
                    print(f"Reached {max_bytes / (1024**3):.2f} GB limit")
                    return
                
                total_bytes += len(chunk)
                yield file_info, chunk
            
            print(f"Completed: {file_info.get('path', file_info['name'])} (Total: {total_bytes / (1024**3):.2f} GB)")
        except Exception as e:
            print(f"Skipping {file_info.get('path', file_info['name'])}: {e}")

In [104]:
def stream_folder_lines(service, folder_id: str,
                        max_bytes: Optional[int] = None,
                        file_types: Optional[list[str]] = None,
                        encoding: str = 'utf-8') -> Generator[tuple[dict, str], None, None]:
    """
    Stream all text files from a folder line by line with optional size limit.
    
    Args:
        service: Google Drive API service object
        folder_id: The ID of the folder to stream from
        max_bytes: Maximum total bytes to stream. None for unlimited.
        file_types: List of file extensions to include (e.g., ['.csv', '.txt'])
        encoding: Text encoding (default: utf-8)
    
    Yields:
        Tuples of (file_metadata, line_string) for each line
    """
    buffer = ''
    current_file = None
    
    for file_info, chunk in stream_folder(service, folder_id, max_bytes, file_types):
        # New file started
        if current_file != file_info['id']:
            # Yield remaining buffer from previous file
            if buffer and current_file:
                yield file_info, buffer
            buffer = ''
            current_file = file_info['id']
        
        buffer += chunk.decode(encoding)
        lines = buffer.split('\n')
        buffer = lines.pop()
        
        for line in lines:
            yield file_info, line
    
    # Yield any remaining content
    if buffer:
        yield file_info, buffer

In [105]:
# Extract folder ID from URL
folder_id = extract_file_id(google_drive_url)
print(f"Folder ID: {folder_id}")

# Verify we can access the folder
try:
    folder_info = service.files().get(fileId=folder_id, fields='id, name, mimeType').execute()
    print(f"Folder name: {folder_info.get('name')}")
    print(f"Type: {folder_info.get('mimeType')}")
except Exception as e:
    print(f"Error accessing folder: {e}")
    print("\nMake sure the folder is shared with your Google account or set to 'Anyone with the link'")

Folder ID: 1atBYNLP5UxUx5keXja5wcUbduYPBD8sn
Folder name: KLAB CHORUS
Type: application/vnd.google-apps.folder


In [106]:
# List all files in the folder (recursively)
print("Files in folder (recursive):")
print("-" * 50)
file_list = []
for file_info in list_files_in_folder(service, folder_id, recursive=True):
    file_list.append(file_info)
    size = int(file_info.get('size', 0))
    size_str = f"{size / 1024**2:.2f} MB" if size > 0 else "N/A"
    print(f"  {file_info.get('path', file_info['name'])} ({size_str})")
print(f"\nTotal: {len(file_list)} files")

Files in folder (recursive):
--------------------------------------------------
  Talks/KLAB TALK abstracts (0.00 MB)
  slack/Knowledge Lab Analytics Jul 1 2025 to Jan 6 2026 - Jan 7 2026.csv (0.02 MB)
  pubs/knowledgelab_publications_abstracts.pdf (0.02 MB)
  C3S2/Copy of The Chicago Center for Computational Social Science (C3S2) - Nov2025 (1.07 MB)
  Grants/Copy of AI Pillar Proposal Summaries (0.01 MB)
  Grants/Copy of New Forms of Socio-Cognitive AI W1 (0.07 MB)
  Grants/Copy of Overview: New Forms of Socio-Cognitive AI (0.01 MB)
  Grants/CVs/.DS_Store (0.01 MB)
  Grants/CVs/CV for James Evans .docx (0.01 MB)
  Grants/CVs/Copy of CV.docx (0.03 MB)
  Grants/CVs/Evans CV Draft.docx (0.03 MB)
  Grants/CVs/Copy of Copy of CV.docx (1.27 MB)
  Grants/CVs/James Evans NIH Biosketch_2024.docx (0.04 MB)
  Grants/CVs/Evans NIH Biosketch 2019.docx (0.04 MB)
  Grants/CVs/biosketch-12-2020-with instructions.docx (0.03 MB)
  Grants/CVs/biosketch-blank-format-rev-10-2021.docx (0.03 MB)
  Grants/NS

## Demo: Stream Data

In [107]:
# Demo 1: Stream all files as raw bytes (up to max_gb limit)
max_bytes = int(max_gb * 1024**3) if max_gb else None

print(f"Streaming up to {max_gb} GB of data...")
print("=" * 50)

total_size = 0
files_streamed = {}

for file_info, chunk in stream_folder(service, folder_id, max_bytes=max_bytes):
    filename = file_info['name']
    if filename not in files_streamed:
        files_streamed[filename] = 0
    files_streamed[filename] += len(chunk)
    total_size += len(chunk)

print("\n" + "=" * 50)
print("Summary:")
for name, size in files_streamed.items():
    print(f"  {name}: {size / 1024**2:.2f} MB")
print(f"\nTotal streamed: {total_size / 1024**2:.2f} MB")

Streaming up to 1 GB of data...
Streaming: Talks/KLAB TALK abstracts
Download progress: 100%
Completed: Talks/KLAB TALK abstracts (Total: 0.00 GB)
Streaming: slack/Knowledge Lab Analytics Jul 1 2025 to Jan 6 2026 - Jan 7 2026.csv
Download progress: 100%
Completed: slack/Knowledge Lab Analytics Jul 1 2025 to Jan 6 2026 - Jan 7 2026.csv (Total: 0.00 GB)
Streaming: pubs/knowledgelab_publications_abstracts.pdf
Download progress: 100%
Completed: pubs/knowledgelab_publications_abstracts.pdf (Total: 0.00 GB)
Streaming: C3S2/Copy of The Chicago Center for Computational Social Science (C3S2) - Nov2025
Download progress: 100%
Completed: C3S2/Copy of The Chicago Center for Computational Social Science (C3S2) - Nov2025 (Total: 0.00 GB)
Streaming: Grants/Copy of AI Pillar Proposal Summaries
Download progress: 100%
Completed: Grants/Copy of AI Pillar Proposal Summaries (Total: 0.00 GB)
Streaming: Grants/Copy of New Forms of Socio-Cognitive AI W1
Download progress: 100%
Completed: Grants/Copy of New 

In [108]:
# Demo 2: Stream text files line by line (preview first 20 lines)
print("Preview of text file contents:")
print("=" * 50)

line_count = 200
max_preview_lines = 20

for file_info, line in stream_folder_lines(service, folder_id, 
                                            max_bytes=50 * 1024**2,  # Limit to 50MB for demo
                                            file_types=['.csv', '.txt', '.json', '.log']):
    line_count += 1
    if line_count <= max_preview_lines:
        display_line = line[:80] + "..." if len(line) > 80 else line
        print(f"[{file_info['name']}] {display_line}")
    elif line_count == max_preview_lines + 1:
        print(f"\n... (showing first {max_preview_lines} lines only)")

print(f"\nTotal lines streamed: {line_count}")

Preview of text file contents:
Streaming: slack/Knowledge Lab Analytics Jul 1 2025 to Jan 6 2026 - Jan 7 2026.csv
Download progress: 100%
Completed: slack/Knowledge Lab Analytics Jul 1 2025 to Jan 6 2026 - Jan 7 2026.csv (Total: 0.00 GB)
Streaming: Events/APTO Retreat/Meeting Recordings/Engler Presentation Transcript.txt
Download progress: 100%
Completed: Events/APTO Retreat/Meeting Recordings/Engler Presentation Transcript.txt (Total: 0.00 GB)

Total lines streamed: 392


In [109]:
file_list

[{'id': '1qCyvWUgmWDToIu33MdhXBei6nc4FIWknhYSryrMNbLY',
  'name': 'KLAB TALK abstracts',
  'mimeType': 'application/vnd.google-apps.document',
  'size': '2762',
  'path': 'Talks/KLAB TALK abstracts'},
 {'id': '1DnEXyQCJJ-F1vqEv1kaWGfE1ZOUaCpbF',
  'name': 'Knowledge Lab Analytics Jul 1 2025 to Jan 6 2026 - Jan 7 2026.csv',
  'mimeType': 'text/csv',
  'size': '23097',
  'path': 'slack/Knowledge Lab Analytics Jul 1 2025 to Jan 6 2026 - Jan 7 2026.csv'},
 {'id': '1iLg58MMwvXf-lyfHOZVZGAjlQ2NZv2xY',
  'name': 'knowledgelab_publications_abstracts.pdf',
  'mimeType': 'application/pdf',
  'size': '18725',
  'path': 'pubs/knowledgelab_publications_abstracts.pdf'},
 {'id': '1kCFlLd--W4Gra5Uje1UWobZKgHHJ38FGDklWRi7B7xI',
  'name': 'Copy of The Chicago Center for Computational Social Science (C3S2) - Nov2025',
  'mimeType': 'application/vnd.google-apps.document',
  'size': '1122155',
  'path': 'C3S2/Copy of The Chicago Center for Computational Social Science (C3S2) - Nov2025'},
 {'id': '13IqZod0j

In [110]:
# Demo 3: Download a single file by name and preview
# Set the filename you want to download (partial match works)
target_filename = 'Copy of New Forms of Socio-Cognitive AI W1'  # Change this to your desired file

# Find the file in file_list
target_file = None
for f in file_list:
    if target_filename.lower() in f.get('name', '').lower():
        target_file = f
        break

if target_file:
    print(f"Downloading: {target_file['name']}")
    print(f"Path: {target_file.get('path', 'N/A')}")
    print(f"Type: {target_file.get('mimeType', 'N/A')}")
    print("-" * 50)
    
    file_data = b''
    for chunk in stream_file_from_drive(service, target_file['id'], 
                                         mime_type=target_file.get('mimeType')):
        file_data += chunk
    
    print(f"Downloaded {len(file_data)} bytes ({len(file_data) / 1024**2:.2f} MB)")
    
    # Preview if it's a text file
    if any(target_file['name'].lower().endswith(ext) for ext in ['.txt', '.csv', '.json', '.log', '.md']) or \
       target_file.get('mimeType') in GOOGLE_DOCS_EXPORT_MIMETYPES:
        print("\nFirst 1000 characters:")
        print("-" * 50)
        print(file_data.decode('utf-8', errors='ignore')[:1000])
else:
    print(f"File '{target_filename}' not found in file_list")
    print("\nAvailable files (first 20):")
    for f in file_list[:20]:
        print(f"  - {f['name']}")

Downloading: Copy of New Forms of Socio-Cognitive AI W1
Path: Grants/Copy of New Forms of Socio-Cognitive AI W1
Type: application/vnd.google-apps.document
--------------------------------------------------
Download progress: 100%
Downloaded 210738 bytes (0.20 MB)

First 1000 characters:
--------------------------------------------------
%PDF-1.4
%
1 0 obj
<</Title (Copy of New Forms of Socio-Cognitive AI W1)
/Producer (Skia/PDF m145 Google Docs Renderer)>>
endobj
3 0 obj
<</ca 1
/BM /Normal>>
endobj
10 0 obj
<</N 3
/Filter /FlateDecode
/Length 296>> stream
e61]shEH	0W'9]Y?bM4	?K.ހe%o${x/@$*Ns˥QSLeZ}K}^'7v!y.'V>s<^(F>7V=fWtV%J-جS#LQ"'IB ENPyzfISPYH)@&/C~{
endstream
endobj
9 0 obj
<</Type /XObject
/Subtype /Image
/Width 1399
/Height 18
/ColorSpace [/ICCBased 10 0 R]
/BitsPerComponent 8
/Filter /FlateDecode
/Length 351>> stream
x1P A[8@(|F&(K6M2"v??~~¨_~? 
 `?mu6_) (/a C>aq5 <KqO~Xp\~
endstream
endobj
11 0 obj
<</Type /XObject
/Subtype /Image
/Widt

## Preprocessing with Unstructured

Use the `unstructured` library to extract and parse content from various file types (PDFs, Word docs, images, etc.)

In [111]:
# Install unstructured (uncomment if needed)
# !pip install unstructured[all-docs]

In [112]:
import tempfile
from pathlib import Path
from unstructured.partition.auto import partition

In [113]:
def parse_file_with_unstructured(service, file_info: dict) -> list:
    """
    Download a file from Google Drive and parse it with unstructured.
    
    Args:
        service: Google Drive API service object
        file_info: File metadata dict with 'id', 'name', 'mimeType'
    
    Returns:
        List of unstructured Element objects
    """
    # Determine file extension
    filename = file_info['name']
    mime_type = file_info.get('mimeType', '')
    
    # Handle Google Docs exports
    if mime_type in GOOGLE_DOCS_EXPORT_MIMETYPES:
        _, ext = GOOGLE_DOCS_EXPORT_MIMETYPES[mime_type]
        filename = filename + ext
    
    # Download file to temp location
    file_data = b''
    for chunk in stream_file_from_drive(service, file_info['id'], mime_type=mime_type):
        file_data += chunk
    
    # Write to temp file and parse
    with tempfile.NamedTemporaryFile(suffix=Path(filename).suffix, delete=False) as tmp:
        tmp.write(file_data)
        tmp_path = tmp.name
    
    try:
        elements = partition(filename=tmp_path)
        return elements
    finally:
        Path(tmp_path).unlink()  # Clean up temp file


def stream_folder_parsed(service, folder_id: str,
                         max_bytes: Optional[int] = None,
                         file_types: Optional[list[str]] = None,
                         recursive: bool = True) -> Generator[tuple[dict, list], None, None]:
    """
    Stream files from a folder and parse each with unstructured.
    
    Args:
        service: Google Drive API service object
        folder_id: The ID of the folder to stream from
        max_bytes: Maximum total bytes to stream
        file_types: List of file extensions to include
        recursive: If True, include nested subfolders
    
    Yields:
        Tuples of (file_metadata, list of unstructured Elements)
    """
    total_bytes = 0
    
    for file_info in list_files_in_folder(service, folder_id, recursive=recursive):
        if file_info.get('mimeType') == 'application/vnd.google-apps.folder':
            continue
        
        if file_types:
            file_name = file_info.get('name', '')
            mime_type = file_info.get('mimeType', '')
            # Check extension or if it's a Google Doc type
            has_ext = any(file_name.lower().endswith(ext.lower()) for ext in file_types)
            is_google_doc = mime_type in GOOGLE_DOCS_EXPORT_MIMETYPES
            if not (has_ext or is_google_doc):
                continue
        
        print(f"Parsing: {file_info.get('path', file_info['name'])}")
        
        try:
            elements = parse_file_with_unstructured(service, file_info)
            
            # Track bytes (approximate from file size)
            file_size = int(file_info.get('size', 0))
            total_bytes += file_size
            
            if max_bytes and total_bytes > max_bytes:
                print(f"Reached {max_bytes / (1024**3):.2f} GB limit")
                yield file_info, elements
                return
            
            print(f"  Extracted {len(elements)} elements")
            yield file_info, elements
            
        except Exception as e:
            print(f"  Error parsing: {e}")

In [114]:
file_list

[{'id': '1qCyvWUgmWDToIu33MdhXBei6nc4FIWknhYSryrMNbLY',
  'name': 'KLAB TALK abstracts',
  'mimeType': 'application/vnd.google-apps.document',
  'size': '2762',
  'path': 'Talks/KLAB TALK abstracts'},
 {'id': '1DnEXyQCJJ-F1vqEv1kaWGfE1ZOUaCpbF',
  'name': 'Knowledge Lab Analytics Jul 1 2025 to Jan 6 2026 - Jan 7 2026.csv',
  'mimeType': 'text/csv',
  'size': '23097',
  'path': 'slack/Knowledge Lab Analytics Jul 1 2025 to Jan 6 2026 - Jan 7 2026.csv'},
 {'id': '1iLg58MMwvXf-lyfHOZVZGAjlQ2NZv2xY',
  'name': 'knowledgelab_publications_abstracts.pdf',
  'mimeType': 'application/pdf',
  'size': '18725',
  'path': 'pubs/knowledgelab_publications_abstracts.pdf'},
 {'id': '1kCFlLd--W4Gra5Uje1UWobZKgHHJ38FGDklWRi7B7xI',
  'name': 'Copy of The Chicago Center for Computational Social Science (C3S2) - Nov2025',
  'mimeType': 'application/vnd.google-apps.document',
  'size': '1122155',
  'path': 'C3S2/Copy of The Chicago Center for Computational Social Science (C3S2) - Nov2025'},
 {'id': '13IqZod0j

In [115]:
# Demo: Parse a single file with unstructured
target_filename = 'KLAB TALK abstracts'  # Change to your file

target_file = None
for f in file_list:
    if target_filename.lower() in f.get('name', '').lower():
        target_file = f
        break

if target_file:
    print(f"Parsing: {target_file['name']}")
    print(f"Type: {target_file.get('mimeType', 'N/A')}")
    print("-" * 50)
    
    elements = parse_file_with_unstructured(service, target_file)
    
    print(f"\nExtracted {len(elements)} elements:")
    print("-" * 50)
    
    # Show element types breakdown
    from collections import Counter
    type_counts = Counter(type(el).__name__ for el in elements)
    print("\nElement types:")
    for el_type, count in type_counts.most_common():
        print(f"  {el_type}: {count}")
    
    # Preview first 10 elements
    print("\nFirst 10 elements:")
    print("-" * 50)
    for i, el in enumerate(elements[:10]):
        text_preview = str(el)[:100] + "..." if len(str(el)) > 100 else str(el)
        print(f"{i+1}. [{type(el).__name__}] {text_preview}")
else:
    print(f"File '{target_filename}' not found")

Parsing: KLAB TALK abstracts
Type: application/vnd.google-apps.document
--------------------------------------------------
Download progress: 100%

Extracted 6 elements:
--------------------------------------------------

Element types:
  NarrativeText: 3
  Title: 2
  Text: 1

First 10 elements:
--------------------------------------------------
1. [NarrativeText] We are thrilled to kick off Cognition Workshop this quarter with a presentation by Dr. James Evans, ...
2. [Title] See you Wednesday, Monica, Akram, Cambria, and Huiqin
3. [Title] Reasoning Models Generate Societies of Thought James Evans, PhD
4. [NarrativeText] Large language models have achieved remarkable capabilities across domains, yet mechanisms underlyin...
5. [NarrativeText] opportunities for agent organization to harness the wisdom of crowds. I further discuss other new fi...
6. [Text] Winter quarter cognition workshop schedule: January 7: Dr. James Evans, Max Palevsky Professor of So...


In [116]:
# Demo: Parse multiple files from folder with unstructured
print("Parsing files from folder...")
print("=" * 50)

all_elements = []
parsed_files = []

# Limit to first 100MB for demo
for file_info, elements in stream_folder_parsed(service, folder_id, 
                                                 max_bytes=100 * 1024**2,
                                                 recursive=True):
    parsed_files.append({
        'name': file_info['name'],
        'path': file_info.get('path', file_info['name']),
        'num_elements': len(elements)
    })
    all_elements.extend(elements)

print("\n" + "=" * 50)
print(f"Summary: Parsed {len(parsed_files)} files, {len(all_elements)} total elements")
print("\nFiles parsed:")
for f in parsed_files:
    print(f"  {f['path']}: {f['num_elements']} elements")

Parsing files from folder...
Parsing: Talks/KLAB TALK abstracts
Download progress: 100%
  Extracted 6 elements
Parsing: slack/Knowledge Lab Analytics Jul 1 2025 to Jan 6 2026 - Jan 7 2026.csv
Download progress: 100%
  Extracted 1 elements
Parsing: pubs/knowledgelab_publications_abstracts.pdf
Download progress: 100%
  Extracted 64 elements
Parsing: C3S2/Copy of The Chicago Center for Computational Social Science (C3S2) - Nov2025
Download progress: 100%
  Extracted 85 elements
Parsing: Grants/Copy of AI Pillar Proposal Summaries
Download progress: 100%
  Extracted 81 elements
Parsing: Grants/Copy of New Forms of Socio-Cognitive AI W1
Download progress: 100%
  Extracted 69 elements
Parsing: Grants/Copy of Overview: New Forms of Socio-Cognitive AI


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Download progress: 100%


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


  Extracted 72 elements
Parsing: Grants/CVs/.DS_Store
Download progress: 100%
  Error parsing: Partitioning is not supported for the FileType.UNK file type.
Parsing: Grants/CVs/CV for James Evans .docx
Download progress: 100%
  Extracted 44 elements
Parsing: Grants/CVs/Copy of CV.docx
Download progress: 100%
  Extracted 91 elements
Parsing: Grants/CVs/Evans CV Draft.docx
Download progress: 100%
  Extracted 84 elements
Parsing: Grants/CVs/Copy of Copy of CV.docx
Download progress: 78%
Download progress: 100%
  Extracted 504 elements
Parsing: Grants/CVs/James Evans NIH Biosketch_2024.docx
Download progress: 100%
  Extracted 54 elements
Parsing: Grants/CVs/Evans NIH Biosketch 2019.docx
Download progress: 100%
  Extracted 71 elements
Parsing: Grants/CVs/biosketch-12-2020-with instructions.docx
Download progress: 100%
  Extracted 42 elements
Parsing: Grants/CVs/biosketch-blank-format-rev-10-2021.docx
Download progress: 100%
  Extracted 11 elements
Parsing: Grants/NSF Files/Copy of APTO-UChi

...

  Extracted 142 elements
Parsing: KLab Asst Folder/NSF Reports/UChicago - NSF-APTO - Newsletter - January 25 - v14.pdf
Download progress: 14%
Download progress: 28%
Download progress: 42%
Download progress: 56%
Download progress: 70%
Download progress: 84%
Download progress: 98%
Download progress: 100%


Cannot set non-stroke color because expected 4 components but got [0.11, 0.063, 0.051]
Cannot set non-stroke color because expected 4 components but got [1]
Cannot set non-stroke color because expected 4 components but got [1]
Cannot set non-stroke color because expected 4 components but got [0.5]


  Extracted 384 elements
Parsing: KLab Asst Folder/NSF Reports/Report for NSF-APTO (01 27 2025 G).pdf
Download progress: 100%
  Extracted 296 elements
Parsing: KLab Asst Folder/.DS_Store
Download progress: 100%
  Error parsing: Partitioning is not supported for the FileType.UNK file type.
Parsing: Events/APTO Retreat/Meeting Recordings/.DS_Store
Download progress: 100%
  Error parsing: Partitioning is not supported for the FileType.UNK file type.
Parsing: Events/APTO Retreat/Meeting Recordings/Introduction Transcript.pdf
Download progress: 100%
  Extracted 283 elements
Parsing: Events/APTO Retreat/Meeting Recordings/Wang Lab Transcript.docx
Download progress: 100%
  Extracted 561 elements
Parsing: Events/APTO Retreat/Meeting Recordings/Engler Presentation Transcript.txt
Download progress: 100%
  Extracted 1 elements
Parsing: Events/APTO Retreat/Meeting Recordings/Introduction Transcript.docx
Download progress: 100%
  Extracted 814 elements
Parsing: Events/APTO Retreat/Meeting Recording

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

Download progress: 100%
  Extracted 150 elements
Parsing: Communications/Website/Old Headshots/Cresposito.png
Download progress: 100%
  Extracted 1 elements
Parsing: Communications/Website/Old Headshots/Wenxuan Shi.jpg
Download progress: 100%
  Extracted 1 elements
Parsing: Communications/Website/Old Headshots/img_2354.jpg
Download progress: 100%
  Extracted 1 elements
Parsing: Communications/Website/Old Headshots/soc_labs50.jpg
Download progress: 58%
Download progress: 100%
  Extracted 1 elements
Parsing: Communications/Website/Old Headshots/2.png
Download progress: 12%
Download progress: 24%
Download progress: 36%
Download progress: 49%
Download progress: 61%
Download progress: 73%
Download progress: 86%
Download progress: 98%
Download progress: 100%
  Extracted 1 elements
Parsing: Communications/Website/Old Headshots/headshot_nl.jpg
Download progress: 100%
  Extracted 1 elements
Parsing: Communications/Website/Old Headshots/img_6319.jpg
Download progress: 60%
Download progress: 100%

In [117]:
# Extract all text content from parsed elements
def safe_str(el) -> str:
    """Safely convert element to string, handling broken __str__ methods."""
    try:
        result = str(el)
        return result if isinstance(result, str) else ""
    except Exception:
        # Try to get text attribute directly
        try:
            return getattr(el, 'text', '') or ""
        except Exception:
            return ""

def elements_to_text(elements: list, separator: str = "\n\n") -> str:
    """Convert unstructured elements to plain text."""
    texts = []
    for el in elements:
        text = safe_str(el)
        if text:  # Only add non-empty strings
            texts.append(text)
    return separator.join(texts)

def elements_to_chunks(elements: list, chunk_size: int = 1000) -> list[str]:
    """Convert elements to text chunks for processing (e.g., embeddings)."""
    chunks = []
    current_chunk = ""
    
    for el in elements:
        text = safe_str(el)
        if not text:  # Skip empty/None elements
            continue
        if len(current_chunk) + len(text) > chunk_size:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = text
        else:
            current_chunk += "\n" + text
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks

# Demo: Convert to text
if all_elements:
    full_text = elements_to_text(all_elements)
    chunks = elements_to_chunks(all_elements, chunk_size=500)
    
    print(f"Total text length: {len(full_text)} characters")
    print(f"Number of chunks (500 char): {len(chunks)}")
    print("\nFirst chunk preview:")
    print("-" * 50)
    print(chunks[0] if chunks else "No chunks")
else:
    print("No elements to process. Run the previous cell first.")

Total text length: 1981894 characters
Number of chunks (500 char): 4201

First chunk preview:
--------------------------------------------------
We are thrilled to kick off Cognition Workshop this quarter with a presentation by Dr. James Evans, Max Palevsky Professor of Sociology and Data Science. Please join us this Wednesday January 7 at 3:30p in BPSB room 122 (IMB conference room) for Dr. Evans’s talk (title and abstract below).
See you Wednesday, Monica, Akram, Cambria, and Huiqin
Reasoning Models Generate Societies of Thought James Evans, PhD


In [118]:
import json
from datetime import datetime

def save_chunks_to_disk(chunks: list[str], 
                        output_path: str = "Data/chunks.json",
                        metadata: dict = None) -> str:
    """
    Save parsed text chunks to disk as JSON.
    
    Args:
        chunks: List of text chunks to save
        output_path: Path to save the JSON file
        metadata: Optional metadata dict (e.g., source folder, parse date)
    
    Returns:
        Path to the saved file
    """
    # Create output directory if needed
    output_dir = Path(output_path).parent
    output_dir.mkdir(parents=True, exist_ok=True)
    
    data = {
        "metadata": {
            "created_at": datetime.now().isoformat(),
            "num_chunks": len(chunks),
            "total_chars": sum(len(c) for c in chunks),
            **(metadata or {})
        },
        "chunks": chunks
    }
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    
    print(f"Saved {len(chunks)} chunks to {output_path}")
    print(f"Total size: {Path(output_path).stat().st_size / 1024:.1f} KB")
    return output_path


def load_chunks_from_disk(input_path: str = "Data/chunks.json") -> tuple[list[str], dict]:
    """
    Load parsed text chunks from disk.
    
    Args:
        input_path: Path to the JSON file
    
    Returns:
        Tuple of (chunks list, metadata dict)
    """
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    chunks = data.get("chunks", [])
    metadata = data.get("metadata", {})
    
    print(f"Loaded {len(chunks)} chunks from {input_path}")
    print(f"Created: {metadata.get('created_at', 'unknown')}")
    print(f"Total chars: {metadata.get('total_chars', 'unknown')}")
    
    return chunks, metadata


def save_parsed_files_to_disk(parsed_files: list[dict],
                               all_elements: list,
                               output_dir: str = "Data/parsed") -> dict:
    """
    Save parsed files with their elements and chunks to disk.
    
    Args:
        parsed_files: List of file metadata dicts from parsing
        all_elements: List of all unstructured elements
        output_dir: Directory to save files
    
    Returns:
        Dict with paths to saved files
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Generate chunks
    chunks = elements_to_chunks(all_elements, chunk_size=500)
    full_text = elements_to_text(all_elements)
    
    # Save chunks
    chunks_path = output_path / "chunks.json"
    save_chunks_to_disk(chunks, str(chunks_path), metadata={
        "source_files": [f['name'] for f in parsed_files],
        "num_files": len(parsed_files)
    })
    
    # Save full text
    text_path = output_path / "full_text.txt"
    with open(text_path, 'w', encoding='utf-8') as f:
        f.write(full_text)
    print(f"Saved full text to {text_path}")
    
    # Save file manifest
    manifest_path = output_path / "manifest.json"
    manifest = {
        "created_at": datetime.now().isoformat(),
        "files": parsed_files,
        "total_elements": len(all_elements),
        "total_chunks": len(chunks),
        "total_chars": len(full_text)
    }
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f"Saved manifest to {manifest_path}")
    
    return {
        "chunks": str(chunks_path),
        "full_text": str(text_path),
        "manifest": str(manifest_path)
    }


def load_parsed_files_from_disk(input_dir: str = "Data/parsed") -> dict:
    """
    Load all parsed data from disk.
    
    Args:
        input_dir: Directory containing saved files
    
    Returns:
        Dict with chunks, full_text, and manifest
    """
    input_path = Path(input_dir)
    
    # Load chunks
    chunks, chunks_metadata = load_chunks_from_disk(str(input_path / "chunks.json"))
    
    # Load full text
    text_path = input_path / "full_text.txt"
    with open(text_path, 'r', encoding='utf-8') as f:
        full_text = f.read()
    print(f"Loaded full text: {len(full_text)} characters")
    
    # Load manifest
    manifest_path = input_path / "manifest.json"
    with open(manifest_path, 'r', encoding='utf-8') as f:
        manifest = json.load(f)
    print(f"Loaded manifest: {manifest['total_elements']} elements from {manifest['num_files'] if 'num_files' in manifest else len(manifest['files'])} files")
    
    return {
        "chunks": chunks,
        "full_text": full_text,
        "manifest": manifest,
        "metadata": chunks_metadata
    }

In [119]:
# Save all parsed data to disk
saved_paths = save_parsed_files_to_disk(parsed_files, all_elements, output_dir="Data/parsed")
print("\nSaved files:")
for key, path in saved_paths.items():
    print(f"  {key}: {path}")

Saved 4201 chunks to Data/parsed/chunks.json
Total size: 1973.0 KB
Saved full text to Data/parsed/full_text.txt
Saved manifest to Data/parsed/manifest.json

Saved files:
  chunks: Data/parsed/chunks.json
  full_text: Data/parsed/full_text.txt
  manifest: Data/parsed/manifest.json


In [120]:
# Demo: Load chunks back from disk
loaded_data = load_parsed_files_from_disk("Data/parsed")

print(f"\nLoaded {len(loaded_data['chunks'])} chunks")
print(f"First chunk preview:")
print("-" * 50)
print(loaded_data['chunks'][0][:500] if loaded_data['chunks'] else "No chunks")

Loaded 4201 chunks from Data/parsed/chunks.json
Created: 2026-01-09T00:38:17.718391
Total chars: 1962006
Loaded full text: 1981894 characters
Loaded manifest: 15561 elements from 103 files

Loaded 4201 chunks
First chunk preview:
--------------------------------------------------
We are thrilled to kick off Cognition Workshop this quarter with a presentation by Dr. James Evans, Max Palevsky Professor of Sociology and Data Science. Please join us this Wednesday January 7 at 3:30p in BPSB room 122 (IMB conference room) for Dr. Evans’s talk (title and abstract below).
See you Wednesday, Monica, Akram, Cambria, and Huiqin
Reasoning Models Generate Societies of Thought James Evans, PhD


## LightRAG: Lightweight Retrieval-Augmented Generation

A simple RAG implementation using:
- **Embeddings**: sentence-transformers for semantic search
- **Vector Store**: FAISS for efficient similarity search
- **LLM**: OpenAI API (or local models) for generation

In [121]:
# Install RAG dependencies (uncomment if needed)
!pip install sentence-transformers faiss-cpu openai


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [122]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
from typing import Optional
import os

class LightRAG:
    """
    Lightweight Retrieval-Augmented Generation system.
    
    Uses sentence-transformers for embeddings and FAISS for vector search.
    """
    
    def __init__(self, 
                 model_name: str = "all-MiniLM-L6-v2",
                 index_path: str = "Data/rag_index"):
        """
        Initialize the RAG system.
        
        Args:
            model_name: HuggingFace model for embeddings
            index_path: Directory to store/load the FAISS index
        """
        self.model_name = model_name
        self.index_path = Path(index_path)
        self.model = None
        self.index = None
        self.chunks = []
        self.embeddings = None
        
    def _load_model(self):
        """Lazy load the embedding model."""
        if self.model is None:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
        return self.model
    
    def build_index(self, chunks: list[str], show_progress: bool = True) -> None:
        """
        Build FAISS index from text chunks.
        
        Args:
            chunks: List of text chunks to index
            show_progress: Whether to show progress bar
        """
        self.chunks = chunks
        model = self._load_model()
        
        print(f"Creating embeddings for {len(chunks)} chunks...")
        self.embeddings = model.encode(
            chunks, 
            show_progress_bar=show_progress,
            convert_to_numpy=True
        )
        
        # Normalize for cosine similarity
        faiss.normalize_L2(self.embeddings)
        
        # Build FAISS index
        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)  # Inner product = cosine similarity after normalization
        self.index.add(self.embeddings)
        
        print(f"Index built: {self.index.ntotal} vectors, dimension {dimension}")
    
    def save_index(self, path: Optional[str] = None) -> str:
        """
        Save the FAISS index and chunks to disk.
        
        Args:
            path: Directory to save files (default: self.index_path)
        
        Returns:
            Path to saved directory
        """
        save_path = Path(path) if path else self.index_path
        save_path.mkdir(parents=True, exist_ok=True)
        
        # Save FAISS index
        faiss.write_index(self.index, str(save_path / "index.faiss"))
        
        # Save chunks and metadata
        with open(save_path / "chunks.pkl", 'wb') as f:
            pickle.dump({
                'chunks': self.chunks,
                'model_name': self.model_name
            }, f)
        
        print(f"Saved index to {save_path}")
        return str(save_path)
    
    def load_index(self, path: Optional[str] = None) -> None:
        """
        Load a previously saved FAISS index.
        
        Args:
            path: Directory containing saved files
        """
        load_path = Path(path) if path else self.index_path
        
        # Load FAISS index
        self.index = faiss.read_index(str(load_path / "index.faiss"))
        
        # Load chunks and metadata
        with open(load_path / "chunks.pkl", 'rb') as f:
            data = pickle.load(f)
            self.chunks = data['chunks']
            self.model_name = data.get('model_name', self.model_name)
        
        print(f"Loaded index: {self.index.ntotal} vectors, {len(self.chunks)} chunks")
    
    def retrieve(self, 
                 query: str, 
                 top_k: int = 5,
                 min_score: float = 0.0) -> list[dict]:
        """
        Retrieve relevant chunks for a query.
        
        Args:
            query: Search query
            top_k: Number of results to return
            min_score: Minimum similarity score (0-1)
        
        Returns:
            List of dicts with 'chunk', 'score', 'index'
        """
        if self.index is None:
            raise ValueError("No index loaded. Call build_index() or load_index() first.")
        
        model = self._load_model()
        
        # Encode query
        query_embedding = model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)
        
        # Search
        scores, indices = self.index.search(query_embedding, top_k)
        
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if score >= min_score and idx < len(self.chunks):
                results.append({
                    'chunk': self.chunks[idx],
                    'score': float(score),
                    'index': int(idx)
                })
        
        return results
    
    def get_context(self, 
                    query: str, 
                    top_k: int = 5,
                    max_tokens: int = 2000) -> str:
        """
        Get formatted context for LLM prompt.
        
        Args:
            query: Search query
            top_k: Number of chunks to retrieve
            max_tokens: Approximate max characters for context
        
        Returns:
            Formatted context string
        """
        results = self.retrieve(query, top_k=top_k)
        
        context_parts = []
        total_chars = 0
        
        for i, result in enumerate(results):
            chunk = result['chunk']
            if total_chars + len(chunk) > max_tokens * 4:  # Rough char-to-token ratio
                break
            context_parts.append(f"[Source {i+1}] (relevance: {result['score']:.2f})\n{chunk}")
            total_chars += len(chunk)
        
        return "\n\n".join(context_parts)
    
    def query(self,
              question: str,
              top_k: int = 5,
              llm_client = None,
              model: str = "gpt-4o-mini",
              system_prompt: str = None) -> dict:
        """
        Full RAG query: retrieve context and generate answer.
        
        Args:
            question: User question
            top_k: Number of chunks to retrieve
            llm_client: OpenAI client (if None, returns context only)
            model: OpenAI model to use
            system_prompt: Custom system prompt
        
        Returns:
            Dict with 'answer', 'context', 'sources'
        """
        # Retrieve relevant chunks
        results = self.retrieve(question, top_k=top_k)
        context = self.get_context(question, top_k=top_k)
        
        if llm_client is None:
            return {
                'answer': None,
                'context': context,
                'sources': results,
                'message': 'No LLM client provided. Set llm_client to generate answers.'
            }
        
        # Build prompt
        if system_prompt is None:
            system_prompt = """You are a helpful assistant that answers questions based on the provided context.
Use only the information from the context to answer. If the context doesn't contain enough information, say so.
Be concise and accurate."""
        
        user_prompt = f"""Context:
{context}

Question: {question}

Please answer based on the context above."""
        
        # Generate response
        response = llm_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7
        )
        
        return {
            'answer': response.choices[0].message.content,
            'context': context,
            'sources': results,
            'model': model
        }

In [123]:
# Initialize LightRAG and build index from loaded chunks
rag = LightRAG(model_name="all-MiniLM-L6-v2", index_path="Data/rag_index")

# Use chunks from loaded_data or from memory
chunks_to_index = loaded_data.get('chunks', chunks)
print(f"Indexing {len(chunks_to_index)} chunks...")

# Build the index
rag.build_index(chunks_to_index)

Indexing 4201 chunks...
Loading embedding model: all-MiniLM-L6-v2
Creating embeddings for 4201 chunks...


Batches:   0%|          | 0/132 [00:00<?, ?it/s]

Index built: 4201 vectors, dimension 384


In [124]:
# Save the index to disk for later use
rag.save_index()

Saved index to Data/rag_index


'Data/rag_index'

In [125]:
# Demo: Test retrieval with a sample query
test_query = "What are the main research topics discussed?"

print(f"Query: {test_query}")
print("=" * 50)

results = rag.retrieve(test_query, top_k=5)

print(f"\nTop {len(results)} relevant chunks:")
print("-" * 50)

for i, result in enumerate(results):
    print(f"\n[{i+1}] Score: {result['score']:.3f}")
    print(f"    {result['chunk'][:200]}...")

Query: What are the main research topics discussed?

Top 5 relevant chunks:
--------------------------------------------------

[1] Score: 0.553
    Research and Teaching Interests...

[2] Score: 0.502
    104 01:05:13.260 --> 01:05:36.750 Alexis Puzon: what papers are associated with and facilitate prediction of new papers, right and new collaborations, then also kind of because people are the vectors ...

[3] Score: 0.499
    literature
1) Follow-up proposal prediction: Given a set of existing literature, predict the proposal of a paper that will cite this set.
2) Author collaboration prediction: Given a set of authors' pu...

[4] Score: 0.499
    talk will discuss a system designed to predict the impacts of newly published research. Given a paper, what will be discovered, invented, and produced? To answer these questions, the system (1) predic...

[5] Score: 0.490
    1)
Follow-up proposal prediction: Given a set of existing literature, predict the proposal of a paper that will cite

In [126]:
# Demo: Get formatted context for LLM
context = rag.get_context(test_query, top_k=3)
print("Formatted context for LLM:")
print("=" * 50)
print(context)

Formatted context for LLM:
[Source 1] (relevance: 0.55)
Research and Teaching Interests

[Source 2] (relevance: 0.50)
104 01:05:13.260 --> 01:05:36.750 Alexis Puzon: what papers are associated with and facilitate prediction of new papers, right and new collaborations, then also kind of because people are the vectors of science. It improves the prediction of new science technology. So and and so maybe, yeah, so there, and then there are approaches that we'll talk about. You know. One is

[Source 3] (relevance: 0.50)
literature
1) Follow-up proposal prediction: Given a set of existing literature, predict the proposal of a paper that will cite this set.
2) Author collaboration prediction: Given a set of authors' publication histories, predict the proposal of a paper that they will go on to collaborate on.
3) Experiment prediction: Given a paper proposal and summaries of papers it cites, predict the list of experiments that will be conducted to test/verify the proposal.


In [127]:
# Optional: Full RAG query with OpenAI
# Uncomment and set your API key to use LLM generation

# from openai import OpenAI
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
# 
# response = rag.query(
#     question="What are the key findings about AI and cognition?",
#     top_k=5,
#     llm_client=client,
#     model="gpt-4o-mini"
# )
# 
# print("Answer:", response['answer'])
# print("\nSources used:", len(response['sources']))

In [128]:
# Helper function: Interactive RAG query (without LLM)
def search_documents(query: str, top_k: int = 5) -> None:
    """Search documents and display results."""
    print(f"🔍 Query: {query}")
    print("=" * 60)
    
    results = rag.retrieve(query, top_k=top_k)
    
    if not results:
        print("No relevant results found.")
        return
    
    for i, result in enumerate(results):
        print(f"\n📄 Result {i+1} (Score: {result['score']:.3f})")
        print("-" * 40)
        # Show full chunk, wrapped
        chunk = result['chunk']
        print(chunk)
        print()

# Try it out
search_documents("workshop presentation")

🔍 Query: workshop presentation

📄 Result 1 (Score: 0.677)
----------------------------------------
Workshop and Communication Channels


📄 Result 2 (Score: 0.560)
----------------------------------------
To catalyze collaborations between these participants and to kick-off the program, The University of Chicago organized a two-day workshop, during November 1 and 2 of 2024, in which researchers across our partner institutions came together, gave lectures, engaged in focused discussions and forged effective working relationships.


📄 Result 3 (Score: 0.522)
----------------------------------------
This intensive workshop ignited a series of productive exchanges regarding methods for predicting and accelerating strategic technology capability, production and use. The workshop helped us begin the process of forming a unified, coherent team that dynamically builds on each other’s strengths and functions as a unified group; like members of a philharmonic orchestra.


📄 Result 4 (Score: 0.504

In [129]:
# Load a previously saved index (for future sessions)
# rag_loaded = LightRAG(index_path="Data/rag_index")
# rag_loaded.load_index()
# search_documents("your query here")  # Uses rag_loaded

## Improved Chunking with Overlap and Metadata

Better chunking strategy that:
1. **Preserves sentence boundaries** - doesn't cut mid-sentence
2. **Adds overlap** - context is preserved across chunk boundaries
3. **Tracks metadata** - source file, element type, position for citations

In [130]:
from dataclasses import dataclass, field
from typing import Optional
import re

@dataclass
class ChunkWithMetadata:
    """A text chunk with associated metadata for RAG retrieval."""
    text: str
    source_file: str = ""
    source_path: str = ""
    element_type: str = ""
    chunk_index: int = 0
    total_chunks: int = 0
    page_number: Optional[int] = None
    section: str = ""
    char_start: int = 0
    char_end: int = 0
    
    def to_dict(self) -> dict:
        """Convert to dictionary for serialization."""
        return {
            'text': self.text,
            'source_file': self.source_file,
            'source_path': self.source_path,
            'element_type': self.element_type,
            'chunk_index': self.chunk_index,
            'total_chunks': self.total_chunks,
            'page_number': self.page_number,
            'section': self.section,
            'char_start': self.char_start,
            'char_end': self.char_end
        }
    
    @classmethod
    def from_dict(cls, d: dict) -> 'ChunkWithMetadata':
        """Create from dictionary."""
        return cls(**d)
    
    def __str__(self) -> str:
        return self.text
    
    def citation(self) -> str:
        """Generate a citation string for this chunk."""
        parts = []
        if self.source_file:
            parts.append(self.source_file)
        if self.page_number:
            parts.append(f"p.{self.page_number}")
        if self.section:
            parts.append(f"§{self.section}")
        return " | ".join(parts) if parts else "Unknown source"

In [131]:
def smart_chunk_with_overlap(text: str,
                              chunk_size: int = 500,
                              overlap: int = 100,
                              min_chunk_size: int = 50) -> list[tuple[str, int, int]]:
    """
    Split text into chunks with overlap, respecting sentence boundaries.
    
    Args:
        text: The text to chunk
        chunk_size: Target size for each chunk (characters)
        overlap: Number of characters to overlap between chunks
        min_chunk_size: Minimum chunk size (avoids tiny final chunks)
    
    Returns:
        List of tuples: (chunk_text, char_start, char_end)
    """
    if not text or len(text) < min_chunk_size:
        return [(text, 0, len(text))] if text else []
    
    # Split into sentences using regex
    # Handles: periods, question marks, exclamation points, followed by space or newline
    sentence_pattern = r'(?<=[.!?])\s+'
    sentences = re.split(sentence_pattern, text)
    
    chunks = []
    current_chunk = ""
    current_start = 0
    char_pos = 0
    
    for i, sentence in enumerate(sentences):
        sentence_len = len(sentence)
        
        # If adding this sentence would exceed chunk_size
        if len(current_chunk) + sentence_len > chunk_size and current_chunk:
            # Save current chunk
            chunk_end = char_pos
            chunks.append((current_chunk.strip(), current_start, chunk_end))
            
            # Start new chunk with overlap
            # Find overlap point (go back 'overlap' characters from end)
            if overlap > 0 and len(current_chunk) > overlap:
                # Find sentence boundary near overlap point
                overlap_start = len(current_chunk) - overlap
                # Try to find a sentence break
                overlap_text = current_chunk[overlap_start:]
                current_chunk = overlap_text + " " + sentence
                current_start = chunk_end - len(overlap_text)
            else:
                current_chunk = sentence
                current_start = char_pos
        else:
            # Add sentence to current chunk
            if current_chunk:
                current_chunk += " " + sentence
            else:
                current_chunk = sentence
                current_start = char_pos
        
        # Track character position
        char_pos += sentence_len + 1  # +1 for space/separator
    
    # Don't forget the last chunk
    if current_chunk.strip():
        if len(current_chunk.strip()) >= min_chunk_size or not chunks:
            chunks.append((current_chunk.strip(), current_start, len(text)))
        elif chunks:
            # Merge tiny final chunk with previous
            prev_text, prev_start, _ = chunks[-1]
            chunks[-1] = (prev_text + " " + current_chunk.strip(), prev_start, len(text))
    
    return chunks


# Demo: Test smart chunking
test_text = """This is the first sentence. This is the second sentence with more content. 
Here's a third sentence that adds detail. And a fourth one for good measure.
Now we start a new paragraph with different information. This sentence continues the thought.
The final sentence wraps up this test text with some concluding remarks."""

print("Smart chunking demo:")
print("=" * 50)
print(f"Original text length: {len(test_text)} chars")
print()

test_chunks = smart_chunk_with_overlap(test_text, chunk_size=150, overlap=50)
for i, (chunk, start, end) in enumerate(test_chunks):
    print(f"Chunk {i+1} [{start}:{end}] ({len(chunk)} chars):")
    print(f"  '{chunk[:80]}...'")
    print()

Smart chunking demo:
Original text length: 319 chars

Chunk 1 [0:152] (151 chars):
  'This is the first sentence. This is the second sentence with more content. Here'...'

Chunk 2 [102:246] (144 chars):
  'at adds detail. And a fourth one for good measure. Now we start a new paragraph ...'

Chunk 3 [196:319] (122 chars):
  'information. This sentence continues the thought. The final sentence wraps up th...'



In [132]:
def elements_to_chunks_with_metadata(
    parsed_files: list[tuple[dict, list]],
    chunk_size: int = 500,
    overlap: int = 100
) -> list[ChunkWithMetadata]:
    """
    Convert parsed file elements to chunks with full metadata.
    
    Args:
        parsed_files: List of (file_info, elements) tuples from stream_folder_parsed
        chunk_size: Target chunk size in characters
        overlap: Overlap between chunks in characters
    
    Returns:
        List of ChunkWithMetadata objects
    """
    all_chunks = []
    
    for file_info, elements in parsed_files:
        file_name = file_info.get('name', 'unknown')
        file_path = file_info.get('path', file_name)
        
        # Track current section (from Title elements)
        current_section = ""
        
        # Collect text from elements, tracking metadata
        file_text = ""
        element_boundaries = []  # (start, end, element_type, page_number)
        
        for el in elements:
            el_type = type(el).__name__
            el_text = safe_str(el)
            
            if not el_text:
                continue
            
            # Update section if this is a Title
            if el_type == 'Title':
                current_section = el_text[:100]  # Truncate long titles
            
            # Try to get page number from element metadata
            page_num = None
            if hasattr(el, 'metadata'):
                page_num = getattr(el.metadata, 'page_number', None)
            
            start_pos = len(file_text)
            file_text += el_text + "\n\n"
            end_pos = len(file_text)
            
            element_boundaries.append({
                'start': start_pos,
                'end': end_pos,
                'type': el_type,
                'page': page_num,
                'section': current_section
            })
        
        if not file_text.strip():
            continue
        
        # Now chunk the file text with overlap
        raw_chunks = smart_chunk_with_overlap(file_text, chunk_size, overlap)
        
        # Create ChunkWithMetadata for each chunk
        for i, (chunk_text, char_start, char_end) in enumerate(raw_chunks):
            # Find which element(s) this chunk overlaps with
            overlapping_elements = [
                eb for eb in element_boundaries
                if eb['start'] < char_end and eb['end'] > char_start
            ]
            
            # Get metadata from overlapping elements
            element_types = list(set(eb['type'] for eb in overlapping_elements))
            pages = [eb['page'] for eb in overlapping_elements if eb['page']]
            sections = [eb['section'] for eb in overlapping_elements if eb['section']]
            
            chunk = ChunkWithMetadata(
                text=chunk_text,
                source_file=file_name,
                source_path=file_path,
                element_type=", ".join(element_types) if element_types else "Text",
                chunk_index=i,
                total_chunks=len(raw_chunks),
                page_number=pages[0] if pages else None,
                section=sections[0] if sections else "",
                char_start=char_start,
                char_end=char_end
            )
            all_chunks.append(chunk)
    
    return all_chunks


# We need to re-run parsing to get the (file_info, elements) pairs
# Let's use a helper that works with existing data
def chunks_to_metadata_chunks(
    old_chunks: list[str],
    source_name: str = "Google Drive",
    chunk_size: int = 500,
    overlap: int = 100
) -> list[ChunkWithMetadata]:
    """
    Convert existing plain chunks to ChunkWithMetadata with re-chunking.
    
    Use this when you have already parsed data but want to add overlap.
    """
    # Join all chunks back together
    full_text = "\n\n".join(old_chunks)
    
    # Re-chunk with overlap
    raw_chunks = smart_chunk_with_overlap(full_text, chunk_size, overlap)
    
    metadata_chunks = []
    for i, (chunk_text, char_start, char_end) in enumerate(raw_chunks):
        chunk = ChunkWithMetadata(
            text=chunk_text,
            source_file=source_name,
            source_path=source_name,
            chunk_index=i,
            total_chunks=len(raw_chunks),
            char_start=char_start,
            char_end=char_end
        )
        metadata_chunks.append(chunk)
    
    return metadata_chunks


print("Conversion functions defined!")

Conversion functions defined!


In [133]:
class LightRAGWithMetadata:
    """
    Enhanced RAG system that tracks source metadata for citations.
    
    Uses sentence-transformers for embeddings and FAISS for vector search,
    with full metadata tracking for each chunk.
    """
    
    def __init__(self, 
                 model_name: str = "all-MiniLM-L6-v2",
                 index_path: str = "Data/rag_index_v2"):
        """
        Initialize the enhanced RAG system.
        
        Args:
            model_name: HuggingFace model for embeddings
            index_path: Directory to store/load the FAISS index
        """
        self.model_name = model_name
        self.index_path = Path(index_path)
        self.model = None
        self.index = None
        self.chunks: list[ChunkWithMetadata] = []
        self.embeddings = None
        
    def _load_model(self):
        """Lazy load the embedding model."""
        if self.model is None:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
        return self.model
    
    def build_index(self, chunks: list[ChunkWithMetadata], show_progress: bool = True) -> None:
        """
        Build FAISS index from ChunkWithMetadata objects.
        
        Args:
            chunks: List of ChunkWithMetadata to index
            show_progress: Whether to show progress bar
        """
        self.chunks = chunks
        model = self._load_model()
        
        # Extract text for embedding
        texts = [chunk.text for chunk in chunks]
        
        print(f"Creating embeddings for {len(chunks)} chunks with metadata...")
        self.embeddings = model.encode(
            texts, 
            show_progress_bar=show_progress,
            convert_to_numpy=True
        )
        
        # Normalize for cosine similarity
        faiss.normalize_L2(self.embeddings)
        
        # Build FAISS index
        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(self.embeddings)
        
        print(f"Index built: {self.index.ntotal} vectors, dimension {dimension}")
        
        # Show metadata stats
        sources = set(c.source_file for c in chunks)
        print(f"Sources indexed: {len(sources)} files")
    
    def save_index(self, path: Optional[str] = None) -> str:
        """Save the FAISS index and metadata chunks to disk."""
        save_path = Path(path) if path else self.index_path
        save_path.mkdir(parents=True, exist_ok=True)
        
        # Save FAISS index
        faiss.write_index(self.index, str(save_path / "index.faiss"))
        
        # Save chunks with metadata
        chunks_data = [chunk.to_dict() for chunk in self.chunks]
        with open(save_path / "chunks_metadata.json", 'w', encoding='utf-8') as f:
            json.dump({
                'model_name': self.model_name,
                'chunks': chunks_data
            }, f, ensure_ascii=False, indent=2)
        
        print(f"Saved index with metadata to {save_path}")
        return str(save_path)
    
    def load_index(self, path: Optional[str] = None) -> None:
        """Load a previously saved FAISS index with metadata."""
        load_path = Path(path) if path else self.index_path
        
        # Load FAISS index
        self.index = faiss.read_index(str(load_path / "index.faiss"))
        
        # Load chunks with metadata
        with open(load_path / "chunks_metadata.json", 'r', encoding='utf-8') as f:
            data = json.load(f)
            self.model_name = data.get('model_name', self.model_name)
            self.chunks = [ChunkWithMetadata.from_dict(c) for c in data['chunks']]
        
        print(f"Loaded index: {self.index.ntotal} vectors, {len(self.chunks)} chunks with metadata")
    
    def retrieve(self, 
                 query: str, 
                 top_k: int = 5,
                 min_score: float = 0.0) -> list[dict]:
        """
        Retrieve relevant chunks with full metadata.
        
        Returns:
            List of dicts with 'chunk' (ChunkWithMetadata), 'score', 'index'
        """
        if self.index is None:
            raise ValueError("No index loaded. Call build_index() or load_index() first.")
        
        model = self._load_model()
        
        # Encode query
        query_embedding = model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)
        
        # Search
        scores, indices = self.index.search(query_embedding, top_k)
        
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if score >= min_score and idx < len(self.chunks):
                results.append({
                    'chunk': self.chunks[idx],
                    'score': float(score),
                    'index': int(idx)
                })
        
        return results
    
    def get_context_with_citations(self, 
                                    query: str, 
                                    top_k: int = 5,
                                    max_tokens: int = 2000) -> tuple[str, list[str]]:
        """
        Get formatted context with citations for LLM prompt.
        
        Returns:
            Tuple of (context_string, list of citations)
        """
        results = self.retrieve(query, top_k=top_k)
        
        context_parts = []
        citations = []
        total_chars = 0
        
        for i, result in enumerate(results):
            chunk: ChunkWithMetadata = result['chunk']
            if total_chars + len(chunk.text) > max_tokens * 4:
                break
            
            citation = chunk.citation()
            citations.append(citation)
            
            context_parts.append(
                f"[Source {i+1}: {citation}] (relevance: {result['score']:.2f})\n{chunk.text}"
            )
            total_chars += len(chunk.text)
        
        return "\n\n".join(context_parts), citations


print("LightRAGWithMetadata class defined!")

LightRAGWithMetadata class defined!


In [134]:
# Demo: Create improved chunks with metadata from existing data
# Using loaded_data from earlier in the notebook

print("Creating improved chunks with overlap and metadata...")
print("=" * 60)

# Convert existing chunks to metadata chunks with overlap
improved_chunks = chunks_to_metadata_chunks(
    loaded_data['chunks'],
    source_name="KLAB CHORUS",
    chunk_size=500,
    overlap=100
)

print(f"Original chunks: {len(loaded_data['chunks'])}")
print(f"Improved chunks (with overlap): {len(improved_chunks)}")

# Show sample chunks with metadata
print("\nSample chunks with metadata:")
print("-" * 60)
for i, chunk in enumerate(improved_chunks[:3]):
    print(f"\nChunk {i+1}:")
    print(f"  Source: {chunk.source_file}")
    print(f"  Position: {chunk.chunk_index + 1}/{chunk.total_chunks}")
    print(f"  Char range: {chunk.char_start}-{chunk.char_end}")
    print(f"  Text preview: {chunk.text[:100]}...")

Creating improved chunks with overlap and metadata...
Original chunks: 4201
Improved chunks (with overlap): 5804

Sample chunks with metadata:
------------------------------------------------------------

Chunk 1:
  Source: KLAB CHORUS
  Position: 1/5804
  Char range: 0-291
  Text preview: We are thrilled to kick off Cognition Workshop this quarter with a presentation by Dr. James Evans, ...

Chunk 2:
  Source: KLAB CHORUS
  Position: 2/5804
  Char range: 191-561
  Text preview: y 7 at 3:30p in BPSB room 122 (IMB conference room) for Dr. Evans’s talk (title and abstract below)....

Chunk 3:
  Source: KLAB CHORUS
  Position: 3/5804
  Char range: 461-798
  Text preview: abilities across domains, yet mechanisms underlying sophisticated reasoning continue to be explored....


In [135]:
# Build the enhanced RAG index with metadata
rag_v2 = LightRAGWithMetadata(
    model_name="all-MiniLM-L6-v2",
    index_path="Data/rag_index_v2"
)

# Build index from improved chunks
rag_v2.build_index(improved_chunks)

# Save for future use
rag_v2.save_index()

Loading embedding model: all-MiniLM-L6-v2
Creating embeddings for 5804 chunks with metadata...


Batches:   0%|          | 0/182 [00:00<?, ?it/s]

Index built: 5804 vectors, dimension 384
Sources indexed: 1 files
Saved index with metadata to Data/rag_index_v2


'Data/rag_index_v2'

In [136]:
# Demo: Search with citations
def search_with_citations(query: str, top_k: int = 5) -> None:
    """Search documents and display results with full citations."""
    print(f"Query: {query}")
    print("=" * 70)
    
    results = rag_v2.retrieve(query, top_k=top_k)
    
    if not results:
        print("No relevant results found.")
        return
    
    for i, result in enumerate(results):
        chunk: ChunkWithMetadata = result['chunk']
        print(f"\nResult {i+1} (Score: {result['score']:.3f})")
        print(f"  Source: {chunk.citation()}")
        print(f"  Chunk: {chunk.chunk_index + 1}/{chunk.total_chunks}")
        print("-" * 50)
        print(chunk.text[:300] + "..." if len(chunk.text) > 300 else chunk.text)
        print()

# Test the search with citations
search_with_citations("AI research collaboration")

Query: AI research collaboration

Result 1 (Score: 0.763)
  Source: KLAB CHORUS
  Chunk: 230/5804
--------------------------------------------------
be imbued with the perspective-taking, creativity, and reasoning required for scientific discovery. One project tests whether AI can generate end-to-end scientific contributions, from hypothesis formation through data collection to interpretation, and examines what emerges when multiple AI agents co...


Result 2 (Score: 0.761)
  Source: KLAB CHORUS
  Chunk: 231/5804
--------------------------------------------------
n, and examines what emerges when multiple AI agents collaborate as a miniature research laboratory. A second project develops automated research platforms that enhance LLMs' capacity for creative hypothesis generation and productive critique, aiming to make AI a genuine intellectual partner in disc...


Result 3 (Score: 0.723)
  Source: KLAB CHORUS
  Chunk: 2076/5804
--------------------------------------------------
vancing 

In [137]:
# Demo: Get context with citations for LLM
query = "What workshops or presentations have been held?"

context, citations = rag_v2.get_context_with_citations(query, top_k=3)

print("Context for LLM (with citations):")
print("=" * 70)
print(context)
print("\n" + "=" * 70)
print("Citations used:")
for i, cite in enumerate(citations, 1):
    print(f"  [{i}] {cite}")

Context for LLM (with citations):
[Source 1: KLAB CHORUS] (relevance: 0.55)
he program, including training models, constructing data representations, and predicting innovation. Following the event, we organized regular weekly meetings for each of these teams to exchange ideas, share progress updates, and preserve the momentum generated by the kickoff workshop. Many professionals attended the workshop including students, post- doctoral research associates, research scientists, university staff, industrial partners, and members of scientific research organizations.

[Source 2: KLAB CHORUS] (relevance: 0.53)
, and the 2023 conference was hosted by the Kellogg School of Management at Northwestern University. These interdisciplinary, in-person only events included panel discussions, keynote addresses, contributed research talks, poster sessions, and networking opportunities. In addition to these formal events for scholarly exchange, both conferences offered catered breaks, lunches, and a co

In [138]:
# Comparison: Original vs Improved RAG
print("Comparison: Original RAG vs Improved RAG with Metadata")
print("=" * 70)

test_query = "machine learning models"

print(f"\nQuery: '{test_query}'")
print("\n" + "-" * 35 + " ORIGINAL RAG " + "-" * 35)
orig_results = rag.retrieve(test_query, top_k=2)
for i, r in enumerate(orig_results):
    print(f"\n[{i+1}] Score: {r['score']:.3f}")
    print(f"    {r['chunk'][:150]}...")

print("\n" + "-" * 30 + " IMPROVED RAG (with metadata) " + "-" * 30)
new_results = rag_v2.retrieve(test_query, top_k=2)
for i, r in enumerate(new_results):
    chunk = r['chunk']
    print(f"\n[{i+1}] Score: {r['score']:.3f}")
    print(f"    Source: {chunk.citation()}")
    print(f"    Chunk {chunk.chunk_index + 1}/{chunk.total_chunks}")
    print(f"    {chunk.text[:150]}...")

Comparison: Original RAG vs Improved RAG with Metadata

Query: 'machine learning models'

----------------------------------- ORIGINAL RAG -----------------------------------

[1] Score: 0.484
    Inventors
Microelectronics/ Semiconductors
Energy Storage
Biotech/Medical
AI Models
Other
UChicago APTO Project Schematic
Models
Outputs
intelligent e...

[2] Score: 0.444
    A foundational principle of the machine learning paradigm is that models can build
their own detailed knowledge bases from training data without havin...

------------------------------ IMPROVED RAG (with metadata) ------------------------------

[1] Score: 0.519
    Source: KLAB CHORUS
    Chunk 942/5804
    Learning. Vol 202. Proceedings of Machine Learning Research. PMLR; 23--29 Jul 2023: 35151-35174. 3. Sourati J, Evans JA. Accelerating science with hum...

[2] Score: 0.494
    Source: KLAB CHORUS
    Chunk 1757/5804
    nd machine learning applications has been driven by the increasing availability of large-scale d

## Hybrid Search and Query Expansion

Further improvements to RAG retrieval:
1. **Hybrid Search** - Combines semantic (embedding) search with keyword (BM25) search for better recall
2. **Query Expansion** - Rewrites queries to improve matching

In [139]:
# Install rank_bm25 for keyword search
!pip install rank_bm25


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [140]:
from rank_bm25 import BM25Okapi
import string

class HybridRAG:
    """
    Hybrid RAG combining semantic search (FAISS) with keyword search (BM25).
    
    This approach improves recall by catching both:
    - Semantically similar content (via embeddings)
    - Exact keyword matches (via BM25)
    """
    
    def __init__(self,
                 model_name: str = "all-MiniLM-L6-v2",
                 index_path: str = "Data/hybrid_rag_index",
                 semantic_weight: float = 0.7):
        """
        Initialize hybrid RAG.
        
        Args:
            model_name: HuggingFace model for embeddings
            index_path: Directory to store/load indexes
            semantic_weight: Weight for semantic vs keyword scores (0-1)
                            0.7 = 70% semantic, 30% keyword
        """
        self.model_name = model_name
        self.index_path = Path(index_path)
        self.semantic_weight = semantic_weight
        self.keyword_weight = 1 - semantic_weight
        
        self.model = None
        self.faiss_index = None
        self.bm25_index = None
        self.chunks: list[ChunkWithMetadata] = []
        self.tokenized_chunks = []
    
    def _load_model(self):
        """Lazy load the embedding model."""
        if self.model is None:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
        return self.model
    
    def _tokenize(self, text: str) -> list[str]:
        """Simple tokenization for BM25."""
        # Lowercase and remove punctuation
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        # Split on whitespace
        tokens = text.split()
        # Remove very short tokens
        tokens = [t for t in tokens if len(t) > 2]
        return tokens
    
    def build_index(self, chunks: list[ChunkWithMetadata], show_progress: bool = True) -> None:
        """
        Build both FAISS and BM25 indexes.
        
        Args:
            chunks: List of ChunkWithMetadata to index
            show_progress: Whether to show progress bar
        """
        self.chunks = chunks
        model = self._load_model()
        
        # Build semantic index (FAISS)
        texts = [chunk.text for chunk in chunks]
        
        print(f"Building semantic index for {len(chunks)} chunks...")
        embeddings = model.encode(texts, show_progress_bar=show_progress, convert_to_numpy=True)
        faiss.normalize_L2(embeddings)
        
        dimension = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dimension)
        self.faiss_index.add(embeddings)
        
        # Build keyword index (BM25)
        print("Building keyword index (BM25)...")
        self.tokenized_chunks = [self._tokenize(text) for text in texts]
        self.bm25_index = BM25Okapi(self.tokenized_chunks)
        
        print(f"Hybrid index built: {self.faiss_index.ntotal} vectors")
    
    def _semantic_search(self, query: str, top_k: int) -> list[tuple[int, float]]:
        """Search using FAISS (semantic similarity)."""
        model = self._load_model()
        query_embedding = model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)
        
        scores, indices = self.faiss_index.search(query_embedding, top_k)
        return [(int(idx), float(score)) for idx, score in zip(indices[0], scores[0]) if idx >= 0]
    
    def _keyword_search(self, query: str, top_k: int) -> list[tuple[int, float]]:
        """Search using BM25 (keyword matching)."""
        query_tokens = self._tokenize(query)
        scores = self.bm25_index.get_scores(query_tokens)
        
        # Get top-k indices
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        # Normalize scores to 0-1 range
        max_score = max(scores) if max(scores) > 0 else 1
        return [(int(idx), float(scores[idx] / max_score)) for idx in top_indices if scores[idx] > 0]
    
    def retrieve(self,
                 query: str,
                 top_k: int = 5,
                 min_score: float = 0.0) -> list[dict]:
        """
        Hybrid retrieval combining semantic and keyword search.
        
        Args:
            query: Search query
            top_k: Number of results to return
            min_score: Minimum combined score
        
        Returns:
            List of dicts with 'chunk', 'score', 'semantic_score', 'keyword_score', 'index'
        """
        # Get more candidates than needed, then re-rank
        candidate_k = top_k * 3
        
        # Semantic search
        semantic_results = self._semantic_search(query, candidate_k)
        semantic_scores = {idx: score for idx, score in semantic_results}
        
        # Keyword search
        keyword_results = self._keyword_search(query, candidate_k)
        keyword_scores = {idx: score for idx, score in keyword_results}
        
        # Combine scores using weighted fusion
        all_indices = set(semantic_scores.keys()) | set(keyword_scores.keys())
        
        combined_results = []
        for idx in all_indices:
            sem_score = semantic_scores.get(idx, 0)
            kw_score = keyword_scores.get(idx, 0)
            
            # Weighted combination
            combined_score = (self.semantic_weight * sem_score + 
                            self.keyword_weight * kw_score)
            
            if combined_score >= min_score:
                combined_results.append({
                    'chunk': self.chunks[idx],
                    'score': combined_score,
                    'semantic_score': sem_score,
                    'keyword_score': kw_score,
                    'index': idx
                })
        
        # Sort by combined score
        combined_results.sort(key=lambda x: x['score'], reverse=True)
        
        return combined_results[:top_k]
    
    def save_index(self, path: Optional[str] = None) -> str:
        """Save both indexes to disk."""
        save_path = Path(path) if path else self.index_path
        save_path.mkdir(parents=True, exist_ok=True)
        
        # Save FAISS index
        faiss.write_index(self.faiss_index, str(save_path / "faiss_index.faiss"))
        
        # Save chunks and BM25 data
        with open(save_path / "hybrid_data.pkl", 'wb') as f:
            pickle.dump({
                'chunks': [c.to_dict() for c in self.chunks],
                'tokenized_chunks': self.tokenized_chunks,
                'model_name': self.model_name,
                'semantic_weight': self.semantic_weight
            }, f)
        
        print(f"Saved hybrid index to {save_path}")
        return str(save_path)
    
    def load_index(self, path: Optional[str] = None) -> None:
        """Load both indexes from disk."""
        load_path = Path(path) if path else self.index_path
        
        # Load FAISS index
        self.faiss_index = faiss.read_index(str(load_path / "faiss_index.faiss"))
        
        # Load chunks and rebuild BM25
        with open(load_path / "hybrid_data.pkl", 'rb') as f:
            data = pickle.load(f)
            self.chunks = [ChunkWithMetadata.from_dict(c) for c in data['chunks']]
            self.tokenized_chunks = data['tokenized_chunks']
            self.model_name = data.get('model_name', self.model_name)
            self.semantic_weight = data.get('semantic_weight', 0.7)
            self.keyword_weight = 1 - self.semantic_weight
        
        # Rebuild BM25 index
        self.bm25_index = BM25Okapi(self.tokenized_chunks)
        
        print(f"Loaded hybrid index: {self.faiss_index.ntotal} vectors")


print("HybridRAG class defined!")

HybridRAG class defined!


In [141]:
class QueryExpander:
    """
    Expands queries to improve retrieval recall.
    
    Techniques:
    1. Synonym expansion - adds related terms
    2. Acronym expansion - expands common acronyms
    3. Multi-query generation - creates variations of the query
    """
    
    # Common synonyms for research/academic domains
    SYNONYMS = {
        'ai': ['artificial intelligence', 'machine learning', 'ml', 'deep learning'],
        'ml': ['machine learning', 'ai', 'artificial intelligence'],
        'nlp': ['natural language processing', 'text processing', 'language models'],
        'llm': ['large language model', 'language model', 'gpt', 'transformer'],
        'research': ['study', 'investigation', 'analysis', 'paper'],
        'paper': ['publication', 'article', 'study', 'research'],
        'model': ['algorithm', 'system', 'method', 'approach'],
        'data': ['dataset', 'information', 'records'],
        'predict': ['forecast', 'estimate', 'anticipate'],
        'analyze': ['examine', 'study', 'investigate', 'evaluate'],
        'workshop': ['seminar', 'conference', 'meeting', 'session'],
        'collaboration': ['partnership', 'cooperation', 'teamwork'],
        'grant': ['funding', 'award', 'proposal'],
        'university': ['institution', 'college', 'academic'],
    }
    
    # Common acronyms
    ACRONYMS = {
        'ai': 'artificial intelligence',
        'ml': 'machine learning',
        'nlp': 'natural language processing',
        'llm': 'large language model',
        'rag': 'retrieval augmented generation',
        'api': 'application programming interface',
        'nsf': 'national science foundation',
        'nih': 'national institutes of health',
        'cv': 'curriculum vitae',
        'phd': 'doctor of philosophy',
    }
    
    def __init__(self, use_synonyms: bool = True, use_acronyms: bool = True):
        self.use_synonyms = use_synonyms
        self.use_acronyms = use_acronyms
    
    def expand_query(self, query: str) -> list[str]:
        """
        Expand a query into multiple related queries.
        
        Args:
            query: Original search query
        
        Returns:
            List of expanded queries (including original)
        """
        queries = [query]
        query_lower = query.lower()
        words = query_lower.split()
        
        # Expand acronyms
        if self.use_acronyms:
            expanded = query_lower
            for acronym, full_form in self.ACRONYMS.items():
                if acronym in words:
                    expanded = expanded.replace(acronym, full_form)
            if expanded != query_lower:
                queries.append(expanded)
        
        # Add synonym variations
        if self.use_synonyms:
            for word in words:
                if word in self.SYNONYMS:
                    for synonym in self.SYNONYMS[word][:2]:  # Limit to 2 synonyms
                        new_query = query_lower.replace(word, synonym)
                        if new_query not in queries:
                            queries.append(new_query)
        
        return queries
    
    def generate_subqueries(self, query: str) -> list[str]:
        """
        Generate focused sub-queries from a complex query.
        
        Args:
            query: Original query
        
        Returns:
            List of sub-queries
        """
        subqueries = [query]
        
        # Split on common question words
        if ' and ' in query.lower():
            parts = query.lower().split(' and ')
            subqueries.extend([p.strip() for p in parts if len(p.strip()) > 3])
        
        # Handle "about X and Y" patterns
        if ' or ' in query.lower():
            parts = query.lower().split(' or ')
            subqueries.extend([p.strip() for p in parts if len(p.strip()) > 3])
        
        return list(set(subqueries))


class HybridRAGWithExpansion(HybridRAG):
    """
    Hybrid RAG with query expansion for improved recall.
    """
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.expander = QueryExpander()
    
    def retrieve_with_expansion(self,
                                 query: str,
                                 top_k: int = 5,
                                 expand: bool = True) -> list[dict]:
        """
        Retrieve with optional query expansion.
        
        Args:
            query: Original query
            top_k: Number of results
            expand: Whether to use query expansion
        
        Returns:
            List of results with combined scores
        """
        if not expand:
            return self.retrieve(query, top_k)
        
        # Expand the query
        expanded_queries = self.expander.expand_query(query)
        
        # Collect results from all queries
        all_results = {}
        
        for i, q in enumerate(expanded_queries):
            # Give original query higher weight
            weight = 1.0 if i == 0 else 0.5
            
            results = self.retrieve(q, top_k=top_k * 2)
            
            for r in results:
                idx = r['index']
                if idx not in all_results:
                    all_results[idx] = {
                        'chunk': r['chunk'],
                        'score': 0,
                        'semantic_score': 0,
                        'keyword_score': 0,
                        'index': idx,
                        'matched_queries': []
                    }
                
                all_results[idx]['score'] += r['score'] * weight
                all_results[idx]['semantic_score'] = max(
                    all_results[idx]['semantic_score'], r['semantic_score']
                )
                all_results[idx]['keyword_score'] = max(
                    all_results[idx]['keyword_score'], r['keyword_score']
                )
                all_results[idx]['matched_queries'].append(q)
        
        # Sort and return top-k
        results = sorted(all_results.values(), key=lambda x: x['score'], reverse=True)
        return results[:top_k]


print("QueryExpander and HybridRAGWithExpansion classes defined!")

QueryExpander and HybridRAGWithExpansion classes defined!


In [142]:
# Demo: Build hybrid RAG index
hybrid_rag = HybridRAGWithExpansion(
    model_name="all-MiniLM-L6-v2",
    index_path="Data/hybrid_rag_index",
    semantic_weight=0.7  # 70% semantic, 30% keyword
)

# Build from improved chunks
hybrid_rag.build_index(improved_chunks)

# Save for later
hybrid_rag.save_index()

Loading embedding model: all-MiniLM-L6-v2
Building semantic index for 5804 chunks...


Batches:   0%|          | 0/182 [00:00<?, ?it/s]

Building keyword index (BM25)...
Hybrid index built: 5804 vectors
Saved hybrid index to Data/hybrid_rag_index


'Data/hybrid_rag_index'

In [143]:
# Demo: Query expansion
expander = QueryExpander()

test_queries = [
    "AI research collaboration",
    "NSF grant proposal",
    "ML model for prediction",
    "workshop presentation slides"
]

print("Query Expansion Demo")
print("=" * 60)

for query in test_queries:
    expanded = expander.expand_query(query)
    print(f"\nOriginal: '{query}'")
    print(f"Expanded ({len(expanded)} variations):")
    for i, eq in enumerate(expanded):
        print(f"  {i+1}. {eq}")

Query Expansion Demo

Original: 'AI research collaboration'
Expanded (7 variations):
  1. AI research collaboration
  2. artificial intelligence research collaboration
  3. machine learning research collaboration
  4. ai study collaboration
  5. ai investigation collaboration
  6. ai research partnership
  7. ai research cooperation

Original: 'NSF grant proposal'
Expanded (4 variations):
  1. NSF grant proposal
  2. national science foundation grant proposal
  3. nsf funding proposal
  4. nsf award proposal

Original: 'ML model for prediction'
Expanded (5 variations):
  1. ML model for prediction
  2. machine learning model for prediction
  3. ai model for prediction
  4. ml algorithm for prediction
  5. ml system for prediction

Original: 'workshop presentation slides'
Expanded (3 variations):
  1. workshop presentation slides
  2. seminar presentation slides
  3. conference presentation slides


In [144]:
# Demo: Hybrid search comparison
def compare_search_methods(query: str, top_k: int = 3):
    """Compare semantic-only, keyword-only, and hybrid search."""
    print(f"Query: '{query}'")
    print("=" * 70)
    
    # Semantic-only (from rag_v2)
    print("\n--- Semantic Search Only ---")
    sem_results = rag_v2.retrieve(query, top_k=top_k)
    for i, r in enumerate(sem_results):
        print(f"[{i+1}] Score: {r['score']:.3f} | {r['chunk'].text[:80]}...")
    
    # Hybrid search
    print("\n--- Hybrid Search (Semantic + Keyword) ---")
    hybrid_results = hybrid_rag.retrieve(query, top_k=top_k)
    for i, r in enumerate(hybrid_results):
        print(f"[{i+1}] Combined: {r['score']:.3f} (sem: {r['semantic_score']:.3f}, kw: {r['keyword_score']:.3f})")
        print(f"     {r['chunk'].text[:80]}...")
    
    # Hybrid with expansion
    print("\n--- Hybrid + Query Expansion ---")
    expanded_results = hybrid_rag.retrieve_with_expansion(query, top_k=top_k)
    for i, r in enumerate(expanded_results):
        print(f"[{i+1}] Combined: {r['score']:.3f}")
        print(f"     Matched queries: {len(r['matched_queries'])}")
        print(f"     {r['chunk'].text[:80]}...")

# Test with a query that benefits from hybrid search
compare_search_methods("NSF APTO funding")

Query: 'NSF APTO funding'

--- Semantic Search Only ---
[1] Score: 0.732 | ia State University
Section 9 - Professor Britta Glennon University of Pennsylva...
[2] Score: 0.726 | of new graduate students, post-doctoral associates, and researchers to work on t...
[3] Score: 0.726 | of new graduate students, post-doctoral associates, and researchers to work on t...

--- Hybrid Search (Semantic + Keyword) ---
[1] Combined: 0.774 (sem: 0.706, kw: 0.934)
     " said Erwin Gianchandani, NSF assistant director for Technology, Innovation and...
[2] Combined: 0.767 (sem: 0.721, kw: 0.873)
     scientists, university staff, industrial partners, and members of scientific res...
[3] Combined: 0.728 (sem: 0.701, kw: 0.793)
     the country. The program received funding in August of 2024 and work began in ea...

--- Hybrid + Query Expansion ---
[1] Combined: 1.011
     Matched queries: 2
     " said Erwin Gianchandani, NSF assistant director for Technology, Innovation and...
[2] Combined: 1.002
     

In [145]:
# Demo: Search with acronym expansion
# This shows how "AI" gets expanded to "artificial intelligence"
compare_search_methods("AI workshop")

Query: 'AI workshop'

--- Semantic Search Only ---
[1] Score: 0.675 | Summaries of 15 Submitted Proposals and October Workshop Pitches
Automated scien...
[2] Score: 0.659 | ture Human Behaviour
2023
AI Societies
How can we harness the emergent creativit...
[3] Score: 0.641 | n, and examines what emerges when multiple AI agents collaborate as a miniature ...

--- Hybrid Search (Semantic + Keyword) ---
[1] Combined: 0.473 (sem: 0.675, kw: 0.000)
     Summaries of 15 Submitted Proposals and October Workshop Pitches
Automated scien...
[2] Combined: 0.462 (sem: 0.659, kw: 0.000)
     ture Human Behaviour
2023
AI Societies
How can we harness the emergent creativit...
[3] Combined: 0.449 (sem: 0.641, kw: 0.000)
     n, and examines what emerges when multiple AI agents collaborate as a miniature ...

--- Hybrid + Query Expansion ---
[1] Combined: 1.112
     Matched queries: 4
     Summaries of 15 Submitted Proposals and October Workshop Pitches
Automated scien...
[2] Combined: 1.072
     Match

In [146]:
# Helper: Full-featured search with all enhancements
def advanced_search(query: str, top_k: int = 5, show_details: bool = True) -> list[dict]:
    """
    Search using the full hybrid RAG with query expansion.
    
    Returns results with citations and score breakdown.
    """
    print(f"Advanced Search: '{query}'")
    print("=" * 70)
    
    # Show expanded queries
    expanded = hybrid_rag.expander.expand_query(query)
    if len(expanded) > 1:
        print(f"Query expanded to {len(expanded)} variations")
    
    # Search
    results = hybrid_rag.retrieve_with_expansion(query, top_k=top_k)
    
    print(f"\nTop {len(results)} results:")
    print("-" * 70)
    
    for i, r in enumerate(results):
        chunk = r['chunk']
        print(f"\nResult {i+1}")
        print(f"  Source: {chunk.citation()}")
        print(f"  Scores: combined={r['score']:.3f}, semantic={r['semantic_score']:.3f}, keyword={r['keyword_score']:.3f}")
        if show_details:
            print(f"  Text: {chunk.text[:200]}...")
    
    return results

# Try it out
results = advanced_search("machine learning prediction models")

Advanced Search: 'machine learning prediction models'

Top 5 results:
----------------------------------------------------------------------

Result 1
  Source: KLAB CHORUS
  Scores: combined=0.652, semantic=0.503, keyword=1.000
  Text: nd machine learning applications has been driven by the increasing availability of large-scale data. In the context of science and technology, data-driven models fed with published scientiﬁc results a...

Result 2
  Source: KLAB CHORUS
  Scores: combined=0.595, semantic=0.459, keyword=0.911
  Text: Learning. Vol 202. Proceedings of Machine Learning Research. PMLR; 23--29 Jul 2023: 35151-35174. 3. Sourati J, Evans JA. Accelerating science with human-aware artificial intelligence. Nat Hum Behav. P...

Result 3
  Source: KLAB CHORUS
  Scores: combined=0.345, semantic=0.493, keyword=0.000
  Text: und, and video that mirror the diverse styles of content represented among their training data14–16. Recent work demonstrates how inference in these models optimiz

In [148]:
def save_all_rag_indexes(output_dir: str = "Data/rag_indexes") -> dict:
    """
    Save all RAG indexes to disk for later use.
    
    Args:
        output_dir: Base directory to save all indexes
    
    Returns:
        Dict with paths to saved indexes
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    saved_paths = {}
    
    print("Saving all RAG indexes...")
    print("=" * 60)
    
    # Save basic RAG
    if rag.index is not None:
        path = str(output_path / "basic")
        rag.save_index(path)
        saved_paths['basic'] = path
        print(f"  Basic RAG: {rag.index.ntotal} vectors")
    
    # Save RAG with metadata
    if rag_v2.index is not None:
        path = str(output_path / "with_metadata")
        rag_v2.save_index(path)
        saved_paths['with_metadata'] = path
        print(f"  RAG with metadata: {rag_v2.index.ntotal} vectors")
    
    # Save hybrid RAG
    if hybrid_rag.faiss_index is not None:
        path = str(output_path / "hybrid")
        hybrid_rag.save_index(path)
        saved_paths['hybrid'] = path
        print(f"  Hybrid RAG: {hybrid_rag.faiss_index.ntotal} vectors")
    
    # Save manifest
    manifest = {
        'saved_at': datetime.now().isoformat(),
        'indexes': saved_paths,
        'total_chunks': len(improved_chunks) if 'improved_chunks' in dir() else 0,
        'model_name': rag.model_name
    }
    
    manifest_path = output_path / "manifest.json"
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    
    print(f"\nManifest saved to {manifest_path}")
    print(f"Total indexes saved: {len(saved_paths)}")
    
    return saved_paths


def load_all_rag_indexes(input_dir: str = "Data/rag_indexes") -> dict:
    """
    Load all RAG indexes from disk.
    
    Args:
        input_dir: Directory containing saved indexes
    
    Returns:
        Dict with loaded RAG instances
    """
    input_path = Path(input_dir)
    
    # Load manifest
    manifest_path = input_path / "manifest.json"
    if manifest_path.exists():
        with open(manifest_path, 'r') as f:
            manifest = json.load(f)
        print(f"Loading indexes saved at {manifest.get('saved_at', 'unknown')}")
    
    loaded_rags = {}
    
    print("Loading RAG indexes...")
    print("=" * 60)
    
    # Load basic RAG
    basic_path = input_path / "basic"
    if basic_path.exists():
        loaded_rag = LightRAG(index_path=str(basic_path))
        loaded_rag.load_index()
        loaded_rags['basic'] = loaded_rag
    
    # Load RAG with metadata
    metadata_path = input_path / "with_metadata"
    if metadata_path.exists():
        loaded_rag_v2 = LightRAGWithMetadata(index_path=str(metadata_path))
        loaded_rag_v2.load_index()
        loaded_rags['with_metadata'] = loaded_rag_v2
    
    # Load hybrid RAG
    hybrid_path = input_path / "hybrid"
    if hybrid_path.exists():
        loaded_hybrid = HybridRAGWithExpansion(index_path=str(hybrid_path))
        loaded_hybrid.load_index()
        loaded_rags['hybrid'] = loaded_hybrid
    
    print(f"\nLoaded {len(loaded_rags)} RAG indexes")
    
    return loaded_rags


print("save_all_rag_indexes() and load_all_rag_indexes() defined!")

save_all_rag_indexes() and load_all_rag_indexes() defined!


In [149]:
# Save all RAG indexes to disk
saved_paths = save_all_rag_indexes("Data/rag_indexes")

print("\nSaved paths:")
for name, path in saved_paths.items():
    print(f"  {name}: {path}")

Saving all RAG indexes...
Saved index to Data/rag_indexes/basic
  Basic RAG: 4201 vectors
Saved index with metadata to Data/rag_indexes/with_metadata
  RAG with metadata: 5804 vectors
Saved hybrid index to Data/rag_indexes/hybrid
  Hybrid RAG: 5804 vectors

Manifest saved to Data/rag_indexes/manifest.json
Total indexes saved: 3

Saved paths:
  basic: Data/rag_indexes/basic
  with_metadata: Data/rag_indexes/with_metadata
  hybrid: Data/rag_indexes/hybrid


In [150]:
# Demo: Load all RAG indexes (for future sessions)
# Uncomment to test loading:

# loaded_rags = load_all_rag_indexes("Data/rag_indexes")
# 
# # Use loaded indexes:
# basic_rag = loaded_rags['basic']
# metadata_rag = loaded_rags['with_metadata']  
# hybrid_rag = loaded_rags['hybrid']
#
# # Search with loaded hybrid RAG
# results = hybrid_rag.retrieve_with_expansion("your query", top_k=5)

In [151]:
# Summary: All RAG implementations available
print("RAG Implementation Summary")
print("=" * 70)
print("""
Available RAG classes in this notebook:

1. LightRAG (basic)
   - Semantic search only (FAISS + sentence-transformers)
   - Simple text chunks without metadata
   - Use: rag.retrieve(query)

2. LightRAGWithMetadata 
   - Semantic search with full source tracking
   - Chunks include: file name, path, section, page number
   - Use: rag_v2.retrieve(query) - returns ChunkWithMetadata
   - Use: rag_v2.get_context_with_citations(query) - returns (context, citations)

3. HybridRAG
   - Combines semantic (FAISS) + keyword (BM25) search
   - Better recall for exact term matches
   - Use: hybrid_rag.retrieve(query) - returns scores for both methods

4. HybridRAGWithExpansion (recommended)
   - All HybridRAG features + query expansion
   - Expands acronyms (AI → artificial intelligence)
   - Adds synonyms (workshop → seminar, conference)
   - Use: hybrid_rag.retrieve_with_expansion(query)

Quick usage:
   results = hybrid_rag.retrieve_with_expansion("your query here", top_k=5)
   for r in results:
       print(f"{r['chunk'].citation()}: {r['chunk'].text[:100]}...")
""")

print(f"\nCurrent indexes loaded:")
print(f"  - rag (basic): {rag.index.ntotal if rag.index else 0} vectors")
print(f"  - rag_v2 (with metadata): {rag_v2.index.ntotal if rag_v2.index else 0} vectors")
print(f"  - hybrid_rag (hybrid + expansion): {hybrid_rag.faiss_index.ntotal if hybrid_rag.faiss_index else 0} vectors")

RAG Implementation Summary

Available RAG classes in this notebook:

1. LightRAG (basic)
   - Semantic search only (FAISS + sentence-transformers)
   - Simple text chunks without metadata
   - Use: rag.retrieve(query)

2. LightRAGWithMetadata 
   - Semantic search with full source tracking
   - Chunks include: file name, path, section, page number
   - Use: rag_v2.retrieve(query) - returns ChunkWithMetadata
   - Use: rag_v2.get_context_with_citations(query) - returns (context, citations)

3. HybridRAG
   - Combines semantic (FAISS) + keyword (BM25) search
   - Better recall for exact term matches
   - Use: hybrid_rag.retrieve(query) - returns scores for both methods

4. HybridRAGWithExpansion (recommended)
   - All HybridRAG features + query expansion
   - Expands acronyms (AI → artificial intelligence)
   - Adds synonyms (workshop → seminar, conference)
   - Use: hybrid_rag.retrieve_with_expansion(query)

Quick usage:
   results = hybrid_rag.retrieve_with_expansion("your query here", 